# Push Model to Hugging Face, Convert to GGUF, and Run with Ollama

This notebook takes the fine-tuned LLaMA 3.2 1B model and:
1. **Pushes the merged model** to Hugging Face Hub
2. **Converts to GGUF** format (Q8_0 quantization) and pushes the GGUF to a separate HF repo
3. **Runs the model locally** with Ollama

In [2]:
import os
import subprocess
import sys
from pathlib import Path

from huggingface_hub import HfApi, login

# --- Configuration ---
HF_USERNAME = "mardonbekhazratov"
MODEL_REPO = f"{HF_USERNAME}/llama-3.2-1b-alpaca"          # full merged model
GGUF_REPO  = f"{HF_USERNAME}/llama-3.2-1b-alpaca-gguf"     # GGUF version

MERGED_MODEL_DIR = "./merged-model"
GGUF_FILE = "llama-3.2-1b-alpaca.Q8_0.gguf"
LLAMA_CPP_DIR = Path("llama.cpp")

## 1. Login to Hugging Face

In [3]:
login()

## 2. Push Merged Model to Hugging Face

Upload the full merged safetensors model (tokenizer + weights) so it can be used directly with `transformers`.

In [3]:
api = HfApi()

# Create repo and upload entire merged-model directory
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=MERGED_MODEL_DIR,
    repo_id=MODEL_REPO,
    repo_type="model",
)
print(f"Merged model pushed to: https://huggingface.co/{MODEL_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Merged model pushed to: https://huggingface.co/mardonbekhazratov/llama-3.2-1b-alpaca


## 3. Convert to GGUF

Use `llama.cpp`'s converter to create a Q8_0 quantized GGUF file from the merged model.

In [4]:
# Install llama.cpp conversion dependencies
reqs = LLAMA_CPP_DIR / "requirements" / "requirements-convert_hf_to_gguf.txt"
if reqs.exists():
    !pip install -r {str(reqs)} -q

In [5]:
# Convert merged model to GGUF (Q8_0 quantization)
converter = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
!python {str(converter)} {MERGED_MODEL_DIR} --outfile {GGUF_FILE} --outtype q8_0

print(f"\nGGUF file created: {GGUF_FILE}")
print(f"Size: {os.path.getsize(GGUF_FILE) / (1024**3):.2f} GB")

INFO:hf-to-gguf:Loading model: merged-model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {8192, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> Q8_0, shape = {2048, 512}
INFO:hf-to-gguf:blk.0.attn_output.weigh

## 4. Push GGUF to Hugging Face

Upload the GGUF file to a separate repo. Ollama can pull directly from HF repos that contain GGUF files.

In [7]:
# Create GGUF repo and upload
api = HfApi()

api.create_repo(repo_id=GGUF_REPO, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj=GGUF_FILE,
    path_in_repo=GGUF_FILE,
    repo_id=GGUF_REPO,
    repo_type="model",
)
print(f"GGUF pushed to: https://huggingface.co/{GGUF_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

GGUF pushed to: https://huggingface.co/mardonbekhazratov/llama-3.2-1b-alpaca-gguf


## 5. Run with Ollama

Ollama can pull GGUF models directly from Hugging Face using the `hf.co/` prefix.

In [ ]:
OLLAMA_MODEL = f"hf.co/{GGUF_REPO}"

# Pull the model from Hugging Face
!ollama pull {OLLAMA_MODEL}

In [ ]:
# Test the model with a prompt
!ollama run {OLLAMA_MODEL} "What is photosynthesis? Answer in 2 sentences."

In [ ]:
# Interactive chat — try more prompts
!ollama run {OLLAMA_MODEL} "Write a Python function that reverses a linked list."

## Summary

| What | Where |
|------|-------|
| Merged model (safetensors) | `https://huggingface.co/mardonbekhazratov/llama-3.2-1b-alpaca` |
| GGUF model | `https://huggingface.co/mardonbekhazratov/llama-3.2-1b-alpaca-gguf` |
| Ollama command | `ollama run hf.co/mardonbekhazratov/llama-3.2-1b-alpaca-gguf` |